# Early FTUE analysis

**Purpose:** 

**Data range:** 

---



## Key Findings

**Retention:**


**Player Level Progression:**


In [1]:
# hide-output
# Import libraries and initialise the BigQuery connector

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

## Aux functions

In [2]:
# hide-output
# Helper functions for weighted progression, percentile calculation, level visualisations, and retention significance
from aux_functions import (
    compute_weighted_progression,
    weighted_quantiles,
    add_event_annotations,
    add_median_lines,
    plot_percentile_comparison,
    compute_retention_significance,
    plot_retention_significance,
)

## Get data

### Player level and game day

In [3]:
# Compute symmetric A/B date windows of equal length anchored on 2026-06-01 (FTUE launch date)

import datetime as dt

new_ftue_date = dt.datetime(2026, 6, 1)
days_from_start = (dt.datetime.today() - new_ftue_date).days
start_date1 = new_ftue_date - dt.timedelta(days=days_from_start)
end_date1 = new_ftue_date-dt.timedelta(days=1)
start_date2 = new_ftue_date
end_date2 = dt.datetime.today()-dt.timedelta(days=1)

# print all dates
print(f"Start Date 1: {start_date1.strftime('%Y-%m-%d')}")
print(f"End Date 1: {end_date1.strftime('%Y-%m-%d')}")
print(f"Start Date 2: {start_date2.strftime('%Y-%m-%d')}")
print(f"End Date 2: {end_date2.strftime('%Y-%m-%d')}")

Start Date 1: 2026-05-08
End Date 1: 2026-05-31
Start Date 2: 2026-06-01
End Date 2: 2026-06-24


In [4]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = False

In [5]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/playerlevel.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 18.16 GB when run.
Estimated query cost: $0.12


In [6]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [7]:
# hide-output
# Preview raw player level and game day data
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version
0,A833E8E02278DA5A,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,6,3,AND,Non-Attributed,0.75.0
1,B968B2DDD4C15233,2026-05-30,2026-05-29,2026-05-24,2026-05-01,1,7,4,AND,Non-Attributed,0.75.0
2,85EFFCDB563952B8,2026-05-29,2026-05-28,2026-05-24,2026-05-01,1,6,4,AND,CPE,0.75.0
3,B583D41CD4513A95,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,2,1,AND,Non-Attributed,0.75.0
4,E5A02364981EA45F,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,3,2,AND,CPE,0.75.0
...,...,...,...,...,...,...,...,...,...,...,...
183977,A8956EAD56732A82,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,4,2,AND,Non-Attributed,0.78.0
183978,6E2D089E110C20AC,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,4,2,IOS,Non-Attributed,0.78.0
183979,F4F48D26DFB9ED7C,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,2,1,AND,Non-Attributed,0.77.0
183980,E446F22E20BB7F75,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,5,3,IOS,Non-Attributed,0.78.0


### Retention

In [8]:
# hide-output
# Estimate query cost for per-install-date retention SQL
query_location = './sql/retention.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [9]:
# hide-output
# Fetch per-install-date retention data from BigQuery or load from local pickle cache
retention_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data = bqc.get(query='./sql/retention.sql', is_path=True, query_parameters=parameters)
    retention_data.to_pickle('./data/retention.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data = pd.read_pickle('./data/retention.pkl')

In [10]:
# hide-output
# Sort retention data and spot-check Android rows
retention_data.sort_values(['install_dt', 'dx','platform'], inplace=True)
retention_data[retention_data['platform'] == 'AND']

,install_dt,dx,platform,cohort_size,retained_size,retention_rate
0,2026-05-08,0,AND,369,369,1.000000
2,2026-05-08,1,AND,369,154,0.417344
4,2026-05-08,3,AND,369,116,0.314363
6,2026-05-08,7,AND,369,96,0.260163
8,2026-05-08,14,AND,369,63,0.170732
...,...,...,...,...,...,...
383,2026-06-22,0,AND,487,487,1.000000
385,2026-06-22,1,AND,487,187,0.383984
386,2026-06-23,0,AND,468,468,1.000000
388,2026-06-23,1,AND,468,189,0.403846


In [11]:
# hide-output
# Estimate query cost for FTUE-split retention SQL (all users)
query_location = './sql/retentiontotal.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [12]:
# hide-output
# Fetch FTUE-split retention (all users) from BigQuery or load from local pickle cache
retention_data_total = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data_total = bqc.get(query='./sql/retentiontotal.sql', is_path=True, query_parameters=parameters)
    retention_data_total.to_pickle('./data/retentiontotal.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total = pd.read_pickle('./data/retentiontotal.pkl')

In [13]:
# hide-output
# Sort FTUE retention data and spot-check at D14
retention_data_total.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total[retention_data_total['dx'] == 14]

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
17,14,AND,A.Pre-FTUE revamp,10,3667,608,0.165803
16,14,AND,B.Post-FTUE revamp,10,4594,693,0.150849
19,14,IOS,A.Pre-FTUE revamp,10,8866,1254,0.141439
18,14,IOS,B.Post-FTUE revamp,10,8334,1317,0.158027


In [14]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/retentiontotalNA.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [15]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
retention_data_total_na = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    retention_data_total_na = bqc.get(query='./sql/retentiontotalNA.sql', is_path=True, query_parameters=parameters)
    retention_data_total_na.to_pickle('./data/retentiontotalNA.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total_na = pd.read_pickle('./data/retentiontotalNA.pkl')

In [16]:
# hide-output
# Sort and preview organic-only FTUE retention data
retention_data_total_na.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total_na

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,24,4135,4135,1.000000
0,0,AND,B.Post-FTUE revamp,24,6106,6106,1.000000
2,0,IOS,A.Pre-FTUE revamp,24,8681,8681,1.000000
3,0,IOS,B.Post-FTUE revamp,24,12064,12064,1.000000
5,1,AND,A.Pre-FTUE revamp,23,4078,1316,0.322707
4,1,AND,B.Post-FTUE revamp,23,5532,1648,0.297903
7,1,IOS,A.Pre-FTUE revamp,23,8636,2878,0.333256
6,1,IOS,B.Post-FTUE revamp,23,11186,3641,0.325496
8,3,AND,A.Pre-FTUE revamp,21,3997,795,0.198899
9,3,AND,B.Post-FTUE revamp,21,5149,858,0.166634


## Process data

In [17]:
# hide-output
# Assign FTUE flag, cap days_since_install to match B.new window, and drop immature cohort rows
dt_mode = 'install_dt'

data['install_dt'] = data[dt_mode]

data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in data['acquisition_type']]

data['FTUE_flag'] = ['B.new' if x >='0.76.0' else 'A.old' for x in data['install_build_version']]

# Making comparison fair
max_dayx_B_new = (pd.to_datetime('today') - pd.to_datetime('2026-06-01')).days
data = data[~(data['days_since_install'] > max_dayx_B_new)]

data.loc[:,'dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
data = data[data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(data['install_dt'])).dt.days - min_days_since_install]

data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,A833E8E02278DA5A,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,6,3,AND,Non-Attributed,0.75.0,N,A.old,dummy
1,B968B2DDD4C15233,2026-05-30,2026-05-29,2026-05-24,2026-05-01,1,7,4,AND,Non-Attributed,0.75.0,N,A.old,dummy
2,85EFFCDB563952B8,2026-05-29,2026-05-28,2026-05-24,2026-05-01,1,6,4,AND,CPE,0.75.0,Y,A.old,dummy
3,B583D41CD4513A95,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,2,1,AND,Non-Attributed,0.75.0,N,A.old,dummy
4,E5A02364981EA45F,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,3,2,AND,CPE,0.75.0,Y,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183977,A8956EAD56732A82,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,4,2,AND,Non-Attributed,0.78.0,N,B.new,dummy
183978,6E2D089E110C20AC,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,4,2,IOS,Non-Attributed,0.78.0,N,B.new,dummy
183979,F4F48D26DFB9ED7C,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,2,1,AND,Non-Attributed,0.77.0,N,B.new,dummy
183980,E446F22E20BB7F75,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,5,3,IOS,Non-Attributed,0.78.0,N,B.new,dummy


In [18]:
# hide-output
# Sanity-check unique user counts per FTUE group
test = data.groupby(['FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

,FTUE_flag,users
0,A.old,17864
1,B.new,22914


## Retention

Retention curve: the share of a cohort's total users who were active on each calendar day since install. Plotted on a **log scale** so that differences between cohorts remain visible at longer horizons where absolute percentages are very small. Apr 2024 D1 retention was ~55%; Apr 2026 has declined to ~51%.

In [19]:
# hide-output
# Add combined dx_platform column for the per-install-date retention line chart
retention_data['combined_dimension'] = retention_data['dx'].astype(str) + '_' + retention_data['platform']
retention_data

,install_dt,dx,platform,cohort_size,retained_size,retention_rate,combined_dimension
0,2026-05-08,0,AND,369,369,1.000000,0_AND
1,2026-05-08,0,IOS,931,931,1.000000,0_IOS
2,2026-05-08,1,AND,369,154,0.417344,1_AND
3,2026-05-08,1,IOS,931,345,0.370569,1_IOS
4,2026-05-08,3,AND,369,116,0.314363,3_AND
...,...,...,...,...,...,...,...
387,2026-06-23,0,IOS,940,940,1.000000,0_IOS
388,2026-06-23,1,AND,468,189,0.403846,1_AND
389,2026-06-23,1,IOS,940,390,0.414894,1_IOS
391,2026-06-24,0,AND,501,501,1.000000,0_AND


In [20]:
# Per-install-date retention rate over time by dx/platform (figure disabled — used for investigation only)
fig = px.line(retention_data[retention_data['dx'] !=0], 
              x='install_dt', 
              y='retention_rate',
              color='combined_dimension',
              title='Retention rate',
              facet_row='platform',
              width=1200,
              height=800,
              hover_data={'install_dt': True, 'retained_size': True},)

#fig.show()

In [21]:
# hide-output
# Preview FTUE-split retention totals (all users)
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
0,0,AND,A.Pre-FTUE revamp,24,7627,7627,1.000000
1,0,AND,B.Post-FTUE revamp,24,10481,10481,1.000000
3,0,IOS,A.Pre-FTUE revamp,24,16035,16035,1.000000
2,0,IOS,B.Post-FTUE revamp,24,20724,20724,1.000000
4,1,AND,A.Pre-FTUE revamp,23,7542,2994,0.396977
5,1,AND,B.Post-FTUE revamp,23,9884,3864,0.390935
6,1,IOS,A.Pre-FTUE revamp,23,15988,6624,0.414311
7,1,IOS,B.Post-FTUE revamp,23,19692,8117,0.412198
8,3,AND,A.Pre-FTUE revamp,21,7359,2066,0.280745
9,3,AND,B.Post-FTUE revamp,21,8966,2353,0.262436


### Cohort sizes

In [22]:
# Bar chart: cohort sizes at each retention checkpoint by FTUE group and platform
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='cohort_size',
    color='FTUE_flag',
    text='cohort_size',
    title='Cohort sizes',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:0}', textposition='outside')
fig.update_layout(
    #yaxis_tickformat='.0%',
    #yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retention rate 

In [23]:
# Bar chart: D1–D21 retention rates by FTUE group and platform (all users)
# Each Dx is tested independently with a two-proportion z-test; Wilson 95% CIs shown as error bars
plot_retention_significance(
    retention_data_total,
    title='Retention rate by FTUE group — all users (95% Wilson CI, per-Dx significance)',
)

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate,ci_low,ci_high,z_stat,p_value,sig_label,n_needed
0,0,AND,A.Pre-FTUE revamp,24,7627,7627,1.000000,0.999497,1.000000,0.000,1.000000,NS,NaN
1,0,AND,B.Post-FTUE revamp,24,10481,10481,1.000000,0.999634,1.000000,0.000,1.000000,NS,NaN
2,0,IOS,A.Pre-FTUE revamp,24,16035,16035,1.000000,0.999760,1.000000,0.000,1.000000,NS,NaN
3,0,IOS,B.Post-FTUE revamp,24,20724,20724,1.000000,0.999815,1.000000,0.000,1.000000,NS,NaN
4,1,AND,A.Pre-FTUE revamp,23,7542,2994,0.396977,0.385990,0.408069,0.809,0.418567,NS,102660.0
5,1,AND,B.Post-FTUE revamp,23,9884,3864,0.390935,0.381359,0.400595,0.809,0.418567,NS,102660.0
6,1,IOS,A.Pre-FTUE revamp,23,15988,6624,0.414311,0.406697,0.421966,0.403,0.686891,NS,852611.0
7,1,IOS,B.Post-FTUE revamp,23,19692,8117,0.412198,0.405341,0.419089,0.403,0.686891,NS,852611.0
8,3,AND,A.Pre-FTUE revamp,21,7359,2066,0.280745,0.270594,0.291124,2.620,0.008801,**,9261.0
9,3,AND,B.Post-FTUE revamp,21,8966,2353,0.262436,0.253432,0.271643,2.620,0.008801,**,9261.0


### Retention rate for Organics

In [24]:
# Bar chart: D1–D21 retention rates by FTUE group and platform (organic / non-attributed only)
# Each Dx is tested independently; small D21 organic cohorts annotated with users needed for significance
plot_retention_significance(
    retention_data_total_na,
    title='Retention rate by FTUE group — organic only (95% Wilson CI, per-Dx significance)',
)

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate,ci_low,ci_high,z_stat,p_value,sig_label,n_needed
0,0,AND,A.Pre-FTUE revamp,24,4135,4135,1.000000,0.999072,1.000000,0.000,1.000000,NS,NaN
1,0,AND,B.Post-FTUE revamp,24,6106,6106,1.000000,0.999371,1.000000,0.000,1.000000,NS,NaN
2,0,IOS,A.Pre-FTUE revamp,24,8681,8681,1.000000,0.999558,1.000000,0.000,1.000000,NS,NaN
3,0,IOS,B.Post-FTUE revamp,24,12064,12064,1.000000,0.999682,1.000000,0.000,1.000000,NS,NaN
4,1,AND,A.Pre-FTUE revamp,23,4078,1316,0.322707,0.308531,0.337217,2.602,0.009264,**,5457.0
5,1,AND,B.Post-FTUE revamp,23,5532,1648,0.297903,0.285995,0.310092,2.602,0.009264,**,5457.0
6,1,IOS,A.Pre-FTUE revamp,23,8636,2878,0.333256,0.323391,0.343270,1.153,0.248874,NS,57579.0
7,1,IOS,B.Post-FTUE revamp,23,11186,3641,0.325496,0.316874,0.334238,1.153,0.248874,NS,57579.0
8,3,AND,A.Pre-FTUE revamp,21,3997,795,0.198899,0.186816,0.211561,3.977,0.000070,***,2249.0
9,3,AND,B.Post-FTUE revamp,21,5149,858,0.166634,0.156705,0.177061,3.977,0.000070,***,2249.0


## Player max level distribution

In [25]:
# Preview player data
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,A833E8E02278DA5A,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,6,3,AND,Non-Attributed,0.75.0,N,A.old,dummy
1,B968B2DDD4C15233,2026-05-30,2026-05-29,2026-05-24,2026-05-01,1,7,4,AND,Non-Attributed,0.75.0,N,A.old,dummy
2,85EFFCDB563952B8,2026-05-29,2026-05-28,2026-05-24,2026-05-01,1,6,4,AND,CPE,0.75.0,Y,A.old,dummy
3,B583D41CD4513A95,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,2,1,AND,Non-Attributed,0.75.0,N,A.old,dummy
4,E5A02364981EA45F,2026-05-30,2026-05-28,2026-05-24,2026-05-01,2,3,2,AND,CPE,0.75.0,Y,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183977,A8956EAD56732A82,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,4,2,AND,Non-Attributed,0.78.0,N,B.new,dummy
183978,6E2D089E110C20AC,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,4,2,IOS,Non-Attributed,0.78.0,N,B.new,dummy
183979,F4F48D26DFB9ED7C,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,2,1,AND,Non-Attributed,0.77.0,N,B.new,dummy
183980,E446F22E20BB7F75,2026-06-24,2026-06-24,2026-06-21,2026-06-01,0,5,3,IOS,Non-Attributed,0.78.0,N,B.new,dummy


In [26]:
# hide-output
# Build level funnel with P10/P50/P90 percentiles per FTUE group, platform, and day since install
pl_ftue_funnel_agg = data.groupby(['max_level','FTUE_flag','platform', 'days_since_install']).agg(
    users=('user_id', 'nunique')
).reset_index()

pl_ftue_funnel_total_agg = pl_ftue_funnel_agg.groupby(['FTUE_flag','platform','days_since_install']).agg(
    total_users=('users', 'sum')
).reset_index()

pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(pl_ftue_funnel_total_agg, on=['FTUE_flag','platform','days_since_install'])
pl_ftue_funnel_agg['pctg_users'] = pl_ftue_funnel_agg['users'] / pl_ftue_funnel_agg['total_users']


pl_ftue_funnel_agg['pctg_diff_users'] = pl_ftue_funnel_agg.groupby(['max_level','platform','days_since_install'])['pctg_users'].pct_change().fillna(0)

pl_ftue_funnel_agg['combined_dimension'] = pl_ftue_funnel_agg['FTUE_flag'].astype(str) + ' | ' + pl_ftue_funnel_agg['days_since_install'].astype(str)

# Calculate percentiles by FTUE_flag, platform, and days_since_install
level_dist = data.groupby(['days_since_install', 'max_level', 'FTUE_flag', 'platform']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts_by_group = level_dist.groupby(['days_since_install', 'FTUE_flag', 'platform']).apply(
    weighted_quantiles, measure_col='max_level', include_groups=False
).reset_index()

# Rename columns for clarity
level_pcts_by_group = level_pcts_by_group.rename(columns={'p10': 'p10_max_level', 'p50': 'p50_max_level', 'p90': 'p90_max_level'})

# Merge into pl_ftue_funnel_agg
pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(
    level_pcts_by_group, 
    on=['days_since_install', 'FTUE_flag', 'platform'], 
    how='left'
)

pl_ftue_funnel_agg = pl_ftue_funnel_agg.loc[pl_ftue_funnel_agg['days_since_install'].isin([0,1,3,7,14,21])]


pl_ftue_funnel_agg

,max_level,FTUE_flag,platform,days_since_install,users,total_users,pctg_users,pctg_diff_users,combined_dimension,p10_max_level,p50_max_level,p90_max_level
0,1,A.old,AND,0,628,5392,0.116469,0.0,A.old | 0,1,4,6
1,1,A.old,AND,1,62,2857,0.021701,0.0,A.old | 1,3,6,8
3,1,A.old,AND,3,25,2007,0.012456,0.0,A.old | 3,5,8,11
7,1,A.old,AND,7,4,1304,0.003067,0.0,A.old | 7,6,11,15
14,1,A.old,AND,14,1,599,0.001669,0.0,A.old | 14,8,14,19
...,...,...,...,...,...,...,...,...,...,...,...,...
2928,155,B.new,AND,3,1,2292,0.000436,0.0,B.new | 3,5,8,11
2931,155,B.new,AND,7,1,1425,0.000702,0.0,B.new | 7,6,11,15
2937,155,B.new,AND,14,1,680,0.001471,0.0,B.new | 14,7,13,20
2943,155,B.new,AND,21,1,159,0.006289,0.0,B.new | 21,9,16,24


In [27]:
# hide-output
# Define level milestone annotations for A.old and B.new FTUE feature unlock points
events_config = {
    'A.old': [
        {'level': 7, 'name': 'SP', 'color':'blue'},
        {'level': 8, 'name': 'Deco', 'color': 'blue'},
        {'level': 10, 'name': 'TimedC', 'color': 'blue'},
        {'level': 16, 'name': 'TA', 'color': 'blue'},
        {'level': 20, 'name': 'GenB', 'color': 'blue'},
        {'level': 25, 'name': 'TgtEvt', 'color': 'blue'},
    ],
    'B.new': [
        {'level': 6, 'name': 'SP', 'color': 'red'},
        {'level': 9, 'name': 'TASign', 'color': 'red'},
        {'level': 10, 'name': 'Deco', 'color': 'red'},
        {'level': 12, 'name': 'TA', 'color': 'red'},
        {'level': 14, 'name': 'GenB', 'color': 'red'},
        {'level': 17, 'name': 'TimedC', 'color': 'red'},
        {'level': 20, 'name': 'TgtEvt', 'color': 'red'},
    ]
}

In [28]:
# Level distribution charts: user counts, percentage share, and day-over-day diff (up to level 30)
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='users',
              color='combined_dimension',
              title='Players level distribution',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)
#fig = add_median_lines(fig, pl_ftue_funnel_agg, x_col='p50_max_level', ftue_col='FTUE_flag', platform_col='platform')

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_users',
              color='combined_dimension',
              title='Players at each level (percentage)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              labels={'max_level': 'Player level'},
              y='pctg_diff_users',
              color='combined_dimension',
              title='Players at each level (percentage diff change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

### Percentile comparison

In [29]:
# Band chart: P10/P50/P90 level progression comparison between A.old and B.new
plot_percentile_comparison(level_pcts_by_group, percentile='all')

# To inspect a single percentile with diff bar:
# plot_percentile_comparison(level_pcts_by_group, percentile='p50')
# plot_percentile_comparison(level_pcts_by_group, percentile='p10')
# plot_percentile_comparison(level_pcts_by_group, percentile='p90')

### Weighted average

For each install cohort, tracks the **weighted average max level** reached as a function of days since install. Weighted average is used to account for varying player counts across level buckets. The percentage-change chart below highlights where the steepest level gains occur in the early-day window.

In [30]:
# hide-output
# Compute weighted average max level by FTUE group, platform, and days since install
days_since_install_baseline = 0

data_filtered = data[data['days_since_install'] >= days_since_install_baseline]

pl_ftue_max_level_agg = compute_weighted_progression(data_filtered, measure_col='max_level', dimension_cols=['dummy', 'days_since_install','FTUE_flag','platform'], min_bucket_size=50)
pl_ftue_max_level_agg['combined_dimension'] = pl_ftue_max_level_agg['dummy'].astype(str) + ' | ' + pl_ftue_max_level_agg['FTUE_flag']

pl_ftue_max_level_agg['pctg_diff_max_level'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['weighted_avg_max_level'].pct_change().fillna(0)

pl_ftue_total_users = pl_ftue_max_level_agg[pl_ftue_max_level_agg['days_since_install'] == 0][['FTUE_flag', 'platform', 'cohort_users']].rename(columns={'cohort_users': 'total_cohort_users'})
pl_ftue_max_level_agg = pl_ftue_max_level_agg.merge(pl_ftue_total_users, on=['FTUE_flag','platform'], how='left')
pl_ftue_max_level_agg['pctg_users'] = pl_ftue_max_level_agg['cohort_users'] / pl_ftue_max_level_agg['total_cohort_users']
pl_ftue_max_level_agg['pctg_diff_users'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['pctg_users'].pct_change().fillna(0)

pl_ftue_max_level_agg

,dummy,days_since_install,FTUE_flag,platform,cohort_users,weighted_avg_max_level,combined_dimension,pctg_diff_max_level,total_cohort_users,pctg_users,pctg_diff_users
0,dummy,0,A.old,AND,5308,3.625185,dummy | A.old,0.000000,5308,1.000000,0.000000
1,dummy,0,A.old,IOS,12301,3.619795,dummy | A.old,0.000000,12301,1.000000,0.000000
2,dummy,0,B.new,AND,7152,3.671749,dummy | B.new,0.012845,7152,1.000000,0.000000
3,dummy,0,B.new,IOS,15548,3.666901,dummy | B.new,0.013014,15548,1.000000,0.000000
4,dummy,1,A.old,AND,2774,5.730837,dummy | A.old,0.000000,5308,0.522607,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
66,dummy,17,A.old,IOS,490,15.456929,dummy | A.old,0.000000,12301,0.039834,0.000000
67,dummy,17,B.new,IOS,318,16.013253,dummy | B.new,0.035992,15548,0.020453,-0.486551
68,dummy,18,A.old,IOS,119,15.626888,dummy | A.old,0.000000,12301,0.009674,0.000000
69,dummy,18,B.new,IOS,154,16.713636,dummy | B.new,0.069543,15548,0.009905,0.023858


In [31]:
# hide-output
# Bar chart: surviving cohort size at each day since install by FTUE group
fig = px.bar(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='cohort_users',
              color='combined_dimension',
              title='Players at each day since install',
              facet_row='platform',
              width=1200,
              height=800,
              barmode='group',
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [32]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [33]:
# hide-output
# Sort data by user and day for per-user progression analysis
data.sort_values(['user_id', 'days_since_install'], inplace=True)
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
119326,10007491E7935123,2026-06-06,2026-06-06,2026-05-31,2026-06-01,0,6,3,IOS,CPE,0.76.0,Y,B.new,dummy
121552,10007491E7935123,2026-06-07,2026-06-06,2026-05-31,2026-06-01,1,6,4,IOS,CPE,0.76.0,Y,B.new,dummy
116878,10007491E7935123,2026-06-08,2026-06-06,2026-05-31,2026-06-01,2,7,4,IOS,CPE,0.76.0,Y,B.new,dummy
121116,10007491E7935123,2026-06-09,2026-06-06,2026-05-31,2026-06-01,3,7,4,IOS,CPE,0.76.0,Y,B.new,dummy
118789,10007491E7935123,2026-06-10,2026-06-06,2026-05-31,2026-06-01,4,8,5,IOS,CPE,0.76.0,Y,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49194,FFFE079DACB6000A,2026-05-17,2026-05-16,2026-05-10,2026-05-01,1,6,4,IOS,UA,0.75.0,N,A.old,dummy
48948,FFFE079DACB6000A,2026-05-19,2026-05-16,2026-05-10,2026-05-01,3,8,5,IOS,UA,0.75.0,N,A.old,dummy
46563,FFFE079DACB6000A,2026-05-20,2026-05-16,2026-05-10,2026-05-01,4,8,5,IOS,UA,0.75.0,N,A.old,dummy
47665,FFFE079DACB6000A,2026-05-22,2026-05-16,2026-05-10,2026-05-01,6,8,5,IOS,UA,0.75.0,N,A.old,dummy


## Game day reached at day x (work in progress)

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [34]:
# hide-output
# Compute weighted average game day progression by install cohort and days since install

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

,install_dt,days_since_install,cohort_users,weighted_avg_max_gameday
0,2026-05-08,0,948,1.954082
1,2026-05-08,1,361,3.484600
2,2026-05-08,2,269,4.763092
3,2026-05-08,3,182,5.623955
4,2026-05-08,4,159,6.368902
...,...,...,...,...
264,2026-06-22,1,465,3.320922
265,2026-06-22,2,320,4.650901
266,2026-06-23,0,1026,2.062847
267,2026-06-23,1,504,3.627451


In [35]:
# hide-output
# Line chart: weighted average game day by install cohort
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [36]:
# hide-output
# Compute and plot P10/P50/P90 game day distribution by days since install
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

In [37]:
# Export notebook to HTML for sharing and archival
export_notebook_html(
    notebook_path='./earlyftue.ipynb',
    output_path='./earlyftue.html',
)

Saved to earlyftue.html


PosixPath('earlyftue.html')